In [17]:
import pandas as pd
import time

In [18]:
# Read in the data
team_games = pd.read_csv('../nba_playoffs_team_games_2010_2024.csv')
player_stats = pd.read_csv('../nba_playoffs_player_stats_2010_2024.csv')
games_metadata = pd.read_csv('../nba_playoffs_games_2010_2024.csv')


In [19]:
# Merge team data with game metadata
team_games_merged = pd.merge(
    team_games,
    games_metadata,
    on=['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION'],  
    suffixes=('_team', '_meta')
)

In [20]:
# Merge player stats with game metadata
player_stats_merged = pd.merge(
    player_stats,
    games_metadata,
    on=['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION'],
    suffixes=('_player', '_meta')
)

In [21]:
team_games_merged

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE_team,MATCHUP_team,WL_team,MIN,FGM,...,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON_team,GAME_DATE_meta,MATCHUP_meta,WL_meta,SEASON_meta
0,42009,1610612749,MIL,Milwaukee Bucks,40900121,2010-04-17,MIL @ ATL,L,240,37,...,14,18,92,-10,0,2009-10,2010-04-17,MIL @ ATL,L,2009-10
1,42009,1610612743,DEN,Denver Nuggets,40900171,2010-04-17,DEN vs. UTA,W,240,48,...,8,25,126,13,0,2009-10,2010-04-17,DEN vs. UTA,W,2009-10
2,42009,1610612741,CHI,Chicago Bulls,40900101,2010-04-17,CHI @ CLE,L,240,37,...,14,19,83,-13,0,2009-10,2010-04-17,CHI @ CLE,L,2009-10
3,42009,1610612762,UTA,Utah Jazz,40900171,2010-04-17,UTA @ DEN,L,240,41,...,10,25,113,-13,0,2009-10,2010-04-17,UTA @ DEN,L,2009-10
4,42009,1610612737,ATL,Atlanta Hawks,40900121,2010-04-17,ATL vs. MIL,W,240,41,...,13,17,102,10,0,2009-10,2010-04-17,ATL vs. MIL,W,2009-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499,42023,1610612738,BOS,Boston Celtics,42300403,2024-06-12,BOS @ DAL,W,240,38,...,9,19,106,7,1,2023-24,2024-06-12,BOS @ DAL,W,2023-24
2500,42023,1610612742,DAL,Dallas Mavericks,42300404,2024-06-14,DAL vs. BOS,W,240,46,...,9,17,122,38,1,2023-24,2024-06-14,DAL vs. BOS,W,2023-24
2501,42023,1610612738,BOS,Boston Celtics,42300404,2024-06-14,BOS @ DAL,L,240,29,...,14,19,84,-38,1,2023-24,2024-06-14,BOS @ DAL,L,2023-24
2502,42023,1610612742,DAL,Dallas Mavericks,42300405,2024-06-17,DAL @ BOS,L,240,35,...,13,20,88,-18,1,2023-24,2024-06-17,DAL @ BOS,L,2023-24


In [22]:
# Rename the ones you want to keep
team_games_merged = team_games_merged.rename(columns={
    'GAME_DATE_meta': 'GAME_DATE',
    'MATCHUP_team': 'MATCHUP',
    'WL_team': 'WL'
})

In [23]:
# Drop the duplicates
team_games_merged = team_games_merged.drop(columns=[
    'GAME_DATE_team', 'MATCHUP_meta', 'WL_meta',
    'SEASON_team', 'SEASON_meta'
], errors='ignore')

In [24]:
team_games_merged['HOME_AWAY'] = team_games_merged['MATCHUP'].apply(
    lambda x: 'Home' if 'vs.' in x else 'Away'
)

In [25]:
team_games_merged['GAME_DATE'] = pd.to_datetime(team_games_merged['GAME_DATE'])
team_games_merged

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,MATCHUP,WL,MIN,FGM,FGA,...,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,GAME_DATE,HOME_AWAY
0,42009,1610612749,MIL,Milwaukee Bucks,40900121,MIL @ ATL,L,240,37,82,...,11,11,1,14,18,92,-10,0,2010-04-17,Away
1,42009,1610612743,DEN,Denver Nuggets,40900171,DEN vs. UTA,W,240,48,84,...,29,6,3,8,25,126,13,0,2010-04-17,Home
2,42009,1610612741,CHI,Chicago Bulls,40900101,CHI @ CLE,L,240,37,87,...,19,12,4,14,19,83,-13,0,2010-04-17,Away
3,42009,1610612762,UTA,Utah Jazz,40900171,UTA @ DEN,L,240,41,75,...,26,4,6,10,25,113,-13,0,2010-04-17,Away
4,42009,1610612737,ATL,Atlanta Hawks,40900121,ATL vs. MIL,W,240,41,76,...,18,8,11,13,17,102,10,0,2010-04-17,Home
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499,42023,1610612738,BOS,Boston Celtics,42300403,BOS @ DAL,W,240,38,82,...,26,4,6,9,19,106,7,1,2024-06-12,Away
2500,42023,1610612742,DAL,Dallas Mavericks,42300404,DAL vs. BOS,W,240,46,91,...,21,7,2,9,17,122,38,1,2024-06-14,Home
2501,42023,1610612738,BOS,Boston Celtics,42300404,BOS @ DAL,L,240,29,80,...,18,2,5,14,19,84,-38,1,2024-06-14,Away
2502,42023,1610612742,DAL,Dallas Mavericks,42300405,DAL @ BOS,L,240,35,78,...,18,4,4,13,20,88,-18,1,2024-06-17,Away


In [26]:
player_stats_merged.head()

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE_player,MATCHUP_player,WL_player,...,PF,PTS,PLUS_MINUS,FANTASY_PTS,VIDEO_AVAILABLE,SEASON_player,GAME_DATE_meta,MATCHUP_meta,WL_meta,SEASON_meta
0,42009,1497,Chauncey Billups,1610612743,DEN,Denver Nuggets,40900171,2010-04-17,DEN vs. UTA,W,...,4,15,3,31.2,0,2009-10,2010-04-17,DEN vs. UTA,W,2009-10
1,42009,2306,Carlos Arroyo,1610612748,MIA,Miami Heat,40900131,2010-04-17,MIA @ BOS,L,...,2,6,0,17.3,0,2009-10,2010-04-17,MIA @ BOS,L,2009-10
2,42009,2457,Jannero Pargo,1610612741,CHI,Chicago Bulls,40900101,2010-04-17,CHI @ CLE,L,...,0,0,1,0.0,0,2009-10,2010-04-17,CHI @ CLE,L,2009-10
3,42009,2754,Tony Allen,1610612738,BOS,Boston Celtics,40900131,2010-04-17,BOS vs. MIA,W,...,1,14,17,30.2,0,2009-10,2010-04-17,BOS vs. MIA,W,2009-10
4,42009,2365,Chris Andersen,1610612743,DEN,Denver Nuggets,40900171,2010-04-17,DEN vs. UTA,W,...,1,0,14,14.7,0,2009-10,2010-04-17,DEN vs. UTA,W,2009-10


In [27]:
player_stats_merged = player_stats_merged.rename(columns={
    'GAME_DATE_player': 'GAME_DATE',
    'MATCHUP_player': 'MATCHUP',
    'WL_player': 'WL'
})

player_stats_merged = player_stats_merged.drop(columns=[
    'GAME_DATE_meta', 'MATCHUP_meta', 'WL_meta',
    'SEASON_player', 'SEASON_meta'
], errors='ignore')

In [28]:
player_stats_merged['HOME_AWAY'] = player_stats_merged['MATCHUP'].apply(
    lambda x: 'Home' if 'vs.' in x else 'Away'
)

In [30]:
player_stats_merged['GAME_DATE'] = pd.to_datetime(player_stats_merged['GAME_DATE'])
player_stats_merged

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,FANTASY_PTS,VIDEO_AVAILABLE,HOME_AWAY
0,42009,1497,Chauncey Billups,1610612743,DEN,Denver Nuggets,40900171,2010-04-17,DEN vs. UTA,W,...,8,2,0,3,4,15,3,31.2,0,Home
1,42009,2306,Carlos Arroyo,1610612748,MIA,Miami Heat,40900131,2010-04-17,MIA @ BOS,L,...,3,1,0,1,2,6,0,17.3,0,Away
2,42009,2457,Jannero Pargo,1610612741,CHI,Chicago Bulls,40900101,2010-04-17,CHI @ CLE,L,...,0,0,0,0,0,0,1,0.0,0,Away
3,42009,2754,Tony Allen,1610612738,BOS,Boston Celtics,40900131,2010-04-17,BOS vs. MIA,W,...,0,3,2,0,1,14,17,30.2,0,Home
4,42009,2365,Chris Andersen,1610612743,DEN,Denver Nuggets,40900171,2010-04-17,DEN vs. UTA,W,...,1,0,2,0,1,0,14,14.7,0,Home
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26359,42023,203957,Danté Exum,1610612742,DAL,Dallas Mavericks,42300405,2024-06-17,DAL @ BOS,L,...,0,0,0,0,0,4,-6,4.0,1,Away
26360,42023,203939,Dwight Powell,1610612742,DAL,Dallas Mavericks,42300405,2024-06-17,DAL @ BOS,L,...,0,0,0,0,0,0,3,1.2,1,Away
26361,42023,203501,Tim Hardaway Jr.,1610612742,DAL,Dallas Mavericks,42300405,2024-06-17,DAL @ BOS,L,...,0,0,0,0,3,0,1,0.0,1,Away
26362,42023,201143,Al Horford,1610612738,BOS,Boston Celtics,42300405,2024-06-17,BOS vs. DAL,W,...,2,2,0,0,3,9,20,28.8,1,Home


In [31]:
team_games_merged.to_csv('team_with_game_metadata.csv', index=False)
player_stats_merged.to_csv('player_with_game_metadata.csv', index=False)